# IFB 09 — Integrated Historical Inference Test

Use raw-ish historical source information to build model-ready features and execute all 48 final model artifacts as one logical process.

In [1]:
# Import libraries
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while (
    not (PROJECT_ROOT / "src").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "inference_feature_builder.yaml"
)

print("Project root:", PROJECT_ROOT)
print("IFB config:", CONFIG_PATH)


Project root: e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk
IFB config: e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\configs\inference_feature_builder.yaml


In [3]:
# Import module for running inference_feature_builder
from src.ontario_peak_risk.inference_feature_builder.io import (
    load_ifb_config,
    resolve_path,
    load_project_history,
)
from src.ontario_peak_risk.inference_feature_builder.builder import (
    build_feature_matrix,
    load_model_contracts,
    feature_contract_report,
    model_feature_frames,
)
from src.ontario_peak_risk.inference_feature_builder.integrated import (
    OperationalIntegratedPipeline,
)


### Define the Origin Time for forecasting
- This is the date from which consumption for the next 24 hours is forecasted.
- This value must be changed according with the requirement

In [ ]:
# Define the Origin time and FSAs
# Origin_Time = "2026-03-25 23:00:00"

In [7]:
# config
config, project_root = load_ifb_config(CONFIG_PATH)
ORIGIN = pd.Timestamp("2025-12-30 23:00:00")
FSAS = list(config["inference"]["fsas"])

history = load_project_history(config, project_root)

In [8]:
# Historical replay does not require an external weather file because current
# frozen models use weather/context at forecast origin, which is present in the
# historical source. This is a smoke test, not a future-weather simulation.
features = build_feature_matrix(
    history,
    pd.DataFrame(),
    ORIGIN,
    FSAS,
    config=config,
)

artifacts = resolve_path(project_root, config["paths"]["artifacts_dir"])
contracts = load_model_contracts(artifacts)

contract = pd.concat(
    [
        feature_contract_report(
            features,
            contracts["rf"],
            model_name="RandomForestRegressor",
        ),
        feature_contract_report(
            features,
            contracts["xgb"],
            model_name="XGBoostClassifier",
        ),
    ],
    ignore_index=True,
)

if contract["status"].eq("FAIL").any():
    display(contract.loc[contract["status"].eq("FAIL")])
    raise ValueError("Feature contract failed; integrated inference stopped.")

frames = model_feature_frames(features, contracts)
pipeline = OperationalIntegratedPipeline(artifacts)
prediction = pipeline.predict_from_feature_frames(frames)




In [9]:
display(prediction.head(30))



,fsa,forecast_origin,target_timestamp,horizon,forecast_consumption_kwh,peak_risk_score,peak_alert
0,L4T,2025-12-30 23:00:00,2025-12-31 00:00:00,1,11391.804060,0.000132,False
1,L4T,2025-12-30 23:00:00,2025-12-31 01:00:00,2,10575.189247,0.000073,False
2,L4T,2025-12-30 23:00:00,2025-12-31 02:00:00,3,9967.082221,0.000057,False
3,L4T,2025-12-30 23:00:00,2025-12-31 03:00:00,4,9662.285006,0.000113,False
4,L4T,2025-12-30 23:00:00,2025-12-31 04:00:00,5,9579.000471,0.000128,False
5,L4T,2025-12-30 23:00:00,2025-12-31 05:00:00,6,9780.371439,0.000157,False
6,L4T,2025-12-30 23:00:00,2025-12-31 06:00:00,7,9966.185314,0.000100,False
7,L4T,2025-12-30 23:00:00,2025-12-31 07:00:00,8,10458.347517,0.000066,False
8,L4T,2025-12-30 23:00:00,2025-12-31 08:00:00,9,11145.486643,0.000220,False
9,L4T,2025-12-30 23:00:00,2025-12-31 09:00:00,10,11855.571180,0.000246,False


In [10]:
print("Rows:", len(prediction))
print("Expected rows:", len(FSAS) * 24)
print("FSA count:", prediction["fsa"].nunique())
print("Horizons:", prediction["horizon"].min(), "to", prediction["horizon"].max())
print("Peak alerts:", int(prediction["peak_alert"].sum()))

Rows: 144
Expected rows: 144
FSA count: 6
Horizons: 1 to 24
Peak alerts: 50


In [12]:

OUTPUT_DIR = (
    project_root
    / config["paths"]["outputs_dir"]
    / "historical_replay"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

features.to_parquet(
    OUTPUT_DIR / "ifb_model_ready_features.parquet",
    index=False,
)
prediction.to_parquet(
    OUTPUT_DIR / "ifb_integrated_prediction.parquet",
    index=False,
)
prediction.to_csv(
    OUTPUT_DIR / "ifb_integrated_prediction.csv",
    index=False,
)

summary_by_fsa = (
    prediction.groupby("fsa", observed=True)
    .agg(
        max_forecast_consumption_kwh=("forecast_consumption_kwh", "max"),
        max_peak_risk_score=("peak_risk_score", "max"),
        peak_alert_hours=("peak_alert", "sum"),
    )
    .reset_index()
)

display(summary_by_fsa)
summary_by_fsa.to_csv(
    OUTPUT_DIR / "ifb_prediction_summary_by_fsa.csv",
    index=False,
)

print("Historical inference outputs saved to:", OUTPUT_DIR)


,fsa,max_forecast_consumption_kwh,max_peak_risk_score,peak_alert_hours
0,L4T,14865.533818,0.415957,6
1,M5R,13135.196049,0.988430,12
2,M5S,6963.333033,0.991410,11
3,M6G,17741.693093,0.951889,6
4,M9R,8212.401477,0.960627,7
5,M9W,18804.563871,0.971639,8


Historical inference outputs saved to: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\outputs\inference_feature_builder\historical_replay
